<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_93_training_trick_ablations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🧪 **Module 93 — Training-Trick Ablations** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 13 · Vision Robustness & Generative Translation


# Module 93 — Training-Trick Ablations

> M83 explained CNN **architectures**. This module is the **empirical sequel** — the regularization tricks that have moved CIFAR-10 / ImageNet accuracy by 2-4 points *each*, all stacked together, without changing the architecture by a single neuron. ndb796's repo ships five ResNet-18 + CIFAR-10 notebooks side by side (Basic · Mixup · LabelSmoothing · Mixup+LabelSmoothing · BatchNorm-Eval) so you can *see* the trick's effect. This module collects the canon: **Mixup, CutMix, AugMix, Manifold-Mixup, CutOut, RandomErasing, Label Smoothing, BN tricks (FRN, GN, SyncBN, EMA), MixUp+CutMix in DeiT/timm, RandAugment, TrivialAugment, EMA weight averaging, SAM, AWP**, plus the 2026 "Bag of Tricks" recipe that gives modern ImageNet accuracy with a ResNet-50 backbone.

### What you'll cover
1. The baseline — clean ResNet-18 CIFAR-10 from M83
2. **Mixup** (Zhang 2017) — linear interpolation of inputs + labels
3. **CutMix** (Yun 2019) — paste a patch + interpolated label
4. **Label Smoothing** (Szegedy 2016) — soft targets
5. **Sample-level** regs — CutOut, RandomErasing, Hide-and-Seek
6. **Manifold-Mixup** (Verma 2019) — mix in hidden space
7. **Modern augmentation pipelines** — RandAugment, TrivialAugment, AugMix, AutoAugment
8. **Batch-Norm tricks** — Sync, Ghost, EMA stats, FRN, GN
9. **Loss-surface smoothers** — SAM, AWP, weight-averaging (SWA, EMA)
10. **The 2026 "Bag of Tricks" recipe** — what every modern training script does


## 1 · Baseline — clean ResNet-18 CIFAR-10

A standard CIFAR-10 baseline (200 epochs, SGD + cosine LR, RandomCrop + HFlip) reaches **~94.3%** top-1. Every trick below is benchmarked against this number on the same architecture, same schedule, same compute.

```
                 baseline  +mixup  +cutmix  +LS    +SAM   +RandAug  EVERYTHING
ResNet-18 CIFAR  94.3      95.0    95.4    94.7   95.2   95.5      96.4
ResNet-50 ImNet  76.5      77.2    77.6    77.1   77.4   77.8      80.4 (BoT)
```

These are not minor improvements — **2-4 points** is the difference between "publishable" and "not publishable" on most leaderboards. And every trick costs **0 inference time** — they're all training-time augmentations or regularizers.

> 🧭 **The principle.** Modern CNNs are **massively over-parameterized** — a ResNet-18 has ~11M parameters and CIFAR-10 has ~50k training images. Without regularization they memorize. The tricks below all do the same intellectual thing: **broaden the empirical distribution the model sees** so it can't memorize, and **smooth the loss surface** so the optimizer finds a flat minimum that generalizes.


## 2 · Mixup — linear interpolation

**Mixup** (Zhang, Cisse, Dauphin, Lopez-Paz, ICLR 2018) is a one-line augmentation:

```
λ ~ Beta(α, α)                          # α ∈ {0.1, 0.2, 0.4, 1.0}
x_mix = λ · x_i + (1−λ) · x_j           # blend pixels
y_mix = λ · y_i + (1−λ) · y_j           # blend one-hot labels
loss = CrossEntropy(model(x_mix), y_mix)
```

That's it. **Pairs of images are blended, labels are blended, cross-entropy is on the blended target.** Effect: model sees infinite virtual examples between the manifold of real images and learns to **linearly interpolate** between classes.

**Why it works (Zhang et al. 2021 follow-up):**
- **Implicit Lipschitz regularization** — forces the model to be smooth between classes
- **Sharper decision boundaries** — Mixup pushes the boundary into the low-density region between class manifolds
- **Calibration** — models trained with Mixup are *better calibrated* (predicted probability ≈ actual accuracy)

**`α` choice.** `α=0.2` for CIFAR-10/100; `α=1.0` for ImageNet. Lower α → more mass near `λ∈{0,1}` (close to no mixing); higher α → more mixing.


In [ ]:
import torch, torch.nn.functional as F
import numpy as np

def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    x_mix = lam * x + (1 - lam) * x[idx]
    return x_mix, y, y[idx], lam

def mixup_loss(pred, y_a, y_b, lam):
    return lam * F.cross_entropy(pred, y_a) + (1 - lam) * F.cross_entropy(pred, y_b)


## 3 · CutMix — paste a patch

**CutMix** (Yun 2019, ICCV best paper) keeps Mixup's label-interpolation idea but in *space* instead of *intensity*:

```
B ~ uniform random box (area ratio = λ)
x_mix = x_i with box replaced by the same box from x_j
y_mix = λ · y_i + (1−λ) · y_j      # exactly proportional to area
```

Why this is better than Mixup for many problems:
- **Pixels stay realistic** — Mixup makes ghostly blends; CutMix is photorealistic
- **Localization signal preserved** — detectors and segmenters can still see real boundaries
- **Stronger when combined with Mixup** — modern recipes do *both* (each with 50% probability)

CutMix beats Mixup by ~0.4 on ImageNet ResNet-50, and the gap widens for detection / segmentation downstream tasks.

> 📦 **Box-sampling math.** Given target area ratio `λ`, sample `cut_w = W·√(1−λ)`, `cut_h = H·√(1−λ)`, then `cx, cy` uniform in `[0, W]×[0, H]`. The "actual" box may clip — recompute `λ` as `1 − (actual box area / total area)` so the label weight is *exact*.


In [ ]:
def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    B, _, H, W = x.shape
    cut_rat = np.sqrt(1.0 - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1 = max(cx - cut_w // 2, 0); y1 = max(cy - cut_h // 2, 0)
    x2 = min(cx + cut_w // 2, W); y2 = min(cy + cut_h // 2, H)
    idx = torch.randperm(B, device=x.device)
    x_mix = x.clone(); x_mix[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam_actual = 1 - ((x2 - x1) * (y2 - y1) / (W * H))   # exact ratio
    return x_mix, y, y[idx], lam_actual


## 4 · Label Smoothing — soft targets

**Label Smoothing** (Szegedy 2016 Inception-v3) replaces the one-hot target `[0, …, 0, 1, 0, …, 0]` with a smoothed version:

```
y_LS[i] = (1 − ε)         if i = true class
        = ε / (K − 1)     otherwise
```

Typically `ε = 0.1` for `K = 1000` ImageNet classes. The cross-entropy loss becomes:

```
L_LS = −Σ y_LS[i] · log p[i]
     = (1 − ε) · L_CE  +  ε · KL(uniform ‖ p)
```

This is a **uniform-class regularizer** — the model is penalized for being too confident on the wrong classes (it's punished for spending zero probability on them).

**Why it works:**
- **Confidence calibration** — the model doesn't drive max-class logit to +∞
- **Mode collapse prevention** in distillation pipelines (M88)
- **Better generalization** — Müller, Kornblith, Hinton (NeurIPS 2019) showed LS implicitly drives feature representations to be more equidistant between classes — but they also showed it *hurts* downstream distillation (which is why pure SFT-distill sometimes uses ε=0)

> 🔢 **Mathematically equivalent perspective.** LS is the same as Cross-Entropy on the original target **plus** a KL term to the uniform distribution. So `L_LS = (1 − ε) · CE + ε · CE_uniform = (1 − ε) · CE + ε · H(uniform, p)`.


In [ ]:
def label_smooth_loss(pred, target, eps=0.1, K=None):
    K = K or pred.size(-1)
    logp = F.log_softmax(pred, dim=-1)
    one_hot = F.one_hot(target, K).float()
    smooth = one_hot * (1 - eps) + eps / (K - 1) * (1 - one_hot)
    return -(smooth * logp).sum(dim=-1).mean()


## 5 · Sample-level regularizers — CutOut, RandomErasing, Hide-and-Seek

Independent of label mixing, these augmentations **delete information** to force the model to use redundant cues.

| Method | Idea | Best use |
|---|---|---|
| **CutOut** (DeVries 2017) | Mask a random rectangle of size `s×s` with mean color | CIFAR — small images |
| **RandomErasing** (Zhong 2017) | Like CutOut but with random aspect, random fill (constant / uniform / per-pixel) | ImageNet — large images |
| **Hide-and-Seek** (Singh 2017) | Divide image into grid; hide random cells | Weakly-supervised localization |
| **GridMask** (Chen 2020) | Regular grid of zero-pixels — preserves structure, removes texture | Detection-friendly |
| **Cow Mask** (French 2020) | Smooth-noise binary mask; preserves arbitrary shapes | Semi-supervised SSL |

**Cutout for CIFAR.** A `16×16` mean-colored square is the standard. Gains ~1.5 points on top of the baseline. **Almost free** computationally.

**A practical note.** CutOut + Mixup + CutMix are *complementary*. Modern recipes do CutOut (or RandomErasing) as a per-image augmentation, then apply Mixup or CutMix *across* the batch. Stacked, they give the largest gain of any single trick combo.


## 6 · Manifold-Mixup — hidden-space mixing

**Manifold Mixup** (Verma et al. 2019) generalizes Mixup to *hidden* layers:

```
1. Sample a layer `k` uniformly from {0 (input), 1, …, L−1}
2. Forward both x_i and x_j through layers [0..k]
3. Mix hidden states: h_mix = λ·h_i + (1−λ)·h_j
4. Forward h_mix through layers [k..L]
5. Cross-entropy on blended labels
```

Why bother:
- **Smoothes the hidden representation manifold** — flatter feature landscape
- **Stronger generalization** than input-space Mixup alone on some benchmarks
- **Better OOD detection** — the network learns "what mixed reasonable inputs look like in feature space" and unseen far-OOD inputs look unlike that

Cost: 2× forward through some layers. Used in: M84-RNN regularization (mix hidden states), large-batch SimCLR pretraining (mix augmented views), MAE-style decoders.

> 🧠 **Connection to AdaIN (M92).** Manifold-Mixup mixes hidden states linearly. AdaIN mixes hidden-state *statistics* (mean + std). The two together define a 2-parameter family: linear in features vs. moment-matching in features. The diffusion-era successors (StyleAlign, IP-Adapter) live on the same axis.


## 7 · Modern augmentation pipelines

By 2019 the field realized hand-tuning augmentations was wasteful. **Learned / search-based augmentation policies:**

| Pipeline | Idea | Search cost |
|---|---|---|
| **AutoAugment** (Cubuk 2019) | RL search over (op, magnitude, prob) sequences | ~5000 GPU-h |
| **RandAugment** (Cubuk 2020) | Drop the search: pick **N** random ops at **magnitude M** uniformly | None |
| **TrivialAugment** (Müller 2021) | Drop **N** too: 1 random op at random magnitude | None |
| **AugMix** (Hendrycks 2020) | Multiple random aug chains, mix the results | None — improves corruption robustness |
| **AutoAugment v2 / TrivialAugmentWide** | Wider magnitude range | None |
| **PixMix** (Hendrycks 2022) | Mix with fractal patterns | None — corruption + calibration boost |

**RandAugment in 6 lines** is the modern default — gives 80% of AutoAugment's gain at 0% of the search cost:

```python
def randaugment(x, N=2, M=9, ops=None):
    ops = ops or AUGS  # rotate, shear-x, shear-y, translate-x, translate-y,
                       # brightness, color, contrast, sharpness, equalize, ...
    for _ in range(N):
        op = random.choice(ops)
        x = op(x, magnitude=M)
    return x
```

`N=2, M=9` is the de-facto setting for ImageNet ResNet-50; gives `+1.0` over baseline + RandomCrop+HFlip.

**The 2026 default in `timm`** is `RandAugment(N=2, M=9) + RandomErasing(p=0.25) + Mixup(α=0.8) + CutMix(α=1.0)` (each of Mixup/CutMix with 50% prob). This is *the* baseline you should benchmark against.


## 8 · Batch-Norm tricks

The vanilla BatchNorm of M83 has a few well-known failure modes. Five fixes that matter:

| Variant | Fix | Best for |
|---|---|---|
| **SyncBN** | Sync statistics across all GPUs (DDP-aware) | Multi-GPU detection / segmentation |
| **Ghost BN / Virtual Batch Norm** | Compute stats on smaller "ghost" sub-batches | Very large batches (8k+) — avoids over-smoothing |
| **EMA BN stats** (a.k.a. "Long-BN") | Use exponentially-moving-average running stats during *training* too, not just eval | More stable late training |
| **FRN** (Filter Response Norm, Singh 2020) | Per-instance / per-channel norm without batch dim | Batch size 1 (style transfer, large images) |
| **GroupNorm** (Wu & He 2018) | Channel groups; no batch dim at all | Small batch (detection, video) |
| **Weight Standardization** | Normalize *weights* layer-wise; pair with GN | Replaces BN's smoothing effect on weights |
| **BN-Eval mode during fine-tune** | Freeze BN at training time after a few epochs | Transfer learning — avoid stat-drift |

**The pattern ndb796's `Batch_Normalization_Evaluation` notebook is showing**: when fine-tuning a pretrained model with **small** batches, freezing BN in eval mode after a few epochs prevents the running stats from drifting under the small-batch noise. Worth ~0.5-1 point on small-batch transfer.

> 🧯 **2026 quick rule.** If batch ≥ 32 and single-GPU → vanilla BN. If multi-GPU → SyncBN. If batch ≤ 8 → GroupNorm. If batch = 1 (style, super-resolution, segmentation w/ large crops) → FRN or LayerNorm. If you're a transformer (ViT, DeiT, ConvNeXt) → LayerNorm always.


## 9 · Loss-surface smoothers — SAM, AWP, weight averaging

The tricks above smooth **input space**. These smooth **parameter space** — they push the optimizer toward flat minima that generalize better.

| Method | Idea | Cost |
|---|---|---|
| **SWA** (Stochastic Weight Averaging, Izmailov 2018) | Average last-N epochs' weights; use the average for inference | ~free |
| **EMA weights** | Exponentially-moving-average of weights `θ_ema = ρ·θ_ema + (1−ρ)·θ`; use ema for eval | ~free; standard in SimCLR, diffusion (M21, M86) |
| **SAM** (Foret 2021 Sharpness-Aware Min) | `min_θ max_{||ε||≤r} L(θ + ε)`; one extra grad step per iter | 2× training time |
| **ASAM / GSAM** | Adaptive / gap-guided SAM variants | 2× |
| **AWP** (Adversarial Weight Perturbation, Wu 2020) | SAM with `r` chosen per-layer | 2× |
| **Lookahead** (Zhang 2019) | Inner-loop fast SGD + outer-loop slow average | ~free |

**SAM in 5 lines** is the most-used:

```python
def sam_step(model, x, y, opt, rho=0.05):
    L = F.cross_entropy(model(x), y); L.backward()
    grad_norm = torch.norm(torch.stack([p.grad.norm() for p in model.parameters()]))
    eps = [rho * p.grad / (grad_norm + 1e-12) for p in model.parameters()]
    for p, e in zip(model.parameters(), eps): p.data.add_(e)         # ascend
    opt.zero_grad(); F.cross_entropy(model(x), y).backward(); opt.step()
    for p, e in zip(model.parameters(), eps): p.data.sub_(e)         # restore
```

**EMA is the everyday default** — `ρ = 0.999` for image classification, `0.9999` for diffusion. Costs almost nothing. Used in DeiT, BEiT, Vision Transformer training, all diffusion training (M21, M86), MAE pretraining, every modern SSL recipe.

> 📐 **Flat-minimum hypothesis (Hochreiter & Schmidhuber 1997 → Keskar 2017).** Small-batch SGD finds flat minima that generalize; large-batch SGD finds sharp minima that don't. SAM, EMA, and SWA all approximate "find a flat minimum" without the wall-clock cost of small batches. This is why every Bag-of-Tricks recipe stacks one of them.


## 10 · The 2026 "Bag of Tricks" recipe

Putting it all together — the canonical 2026 ImageNet training recipe (used by `timm`, DeiT, ConvNeXt, EVA, BEiT):

```
Augmentation:    RandomResizedCrop(224) + HFlip
                 RandAugment(N=2, M=9)
                 RandomErasing(p=0.25)
                 Mixup(α=0.8) ⊕ CutMix(α=1.0) (50/50)
Label:           Label Smoothing(ε=0.1)
Optimizer:       AdamW(wd=0.05, betas=(0.9,0.999))
                 Cosine LR + 5-epoch linear warmup
                 Layer-wise LR decay (transformer)
                 EMA(ρ=0.9999)
Normalization:   LayerNorm (transformer) or SyncBN (CNN multi-GPU)
Reg:             Stochastic Depth (Huang 2016) p=0.1
                 DropPath in transformer
Loss:            CE + Label-Smoothing (already counted)
Schedule:        300 epochs (small models) / 100 (large)
Eval:            EMA weights, TTA optional (10-crop, multi-scale)
```

**With this recipe, ResNet-50 on ImageNet goes from the 2015 baseline 76.5% → 80.4%** (He 2018 "Bag of Tricks") and ConvNeXt-S from 82.1% → 83.6%. **All without changing a single architectural neuron.**

> 🧰 **The lesson for production.** When your test accuracy is one point shy of the goal, the answer is **almost never** "add layers" or "use a bigger model" — it's "apply the Bag of Tricks to the model you already have." Modern ML engineering is much more about *recipe quality* than architecture choice. Each trick is 0.2-1.0 points; stacked, they're 3-4 points; with EMA and SAM, 4-5.

## ✅ Recap

- **Mixup** linearly interpolates inputs + labels — implicit Lipschitz regularization + better calibration.
- **CutMix** does it in space, not intensity — preserves localization signal; combine 50/50 with Mixup.
- **Label Smoothing (ε=0.1)** penalizes overconfidence — better calibration; turn OFF for distillation.
- **CutOut / RandomErasing** delete a patch — almost free; complementary to Mixup.
- **Manifold-Mixup** does the same in hidden space — flatter representation manifold.
- **RandAugment(N=2, M=9)** is the default modern augmentation pipeline; AutoAugment is the search-cost baseline.
- **BN tricks**: SyncBN for multi-GPU, GN for small batch, FRN/LayerNorm for batch-1, freeze in eval mode for fine-tune.
- **SAM, EMA, SWA** smooth the loss surface in parameter space; EMA is the everyday default.
- **2026 Bag of Tricks** stacks all of these — ResNet-50 from 76.5 → 80.4 *with no architecture change*.

Next: **M94 — Conditional Image Translation (Pix2Pix · CycleGAN · StarGAN)**.
